# Caught & Shared — Heterogeneous Graph Approach
**Demonstration of Controllable Multi-Mentor Influence**

This notebook shows how to inject explicit mentor influence into a recommender system using a heterogeneous graph.

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from torch_geometric.data import HeteroData
from torch_geometric.nn import RGCNConv, HeteroConv
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import seaborn as sns

np.random.seed(42)
torch.manual_seed(42)

#### 1. Synthetic Data Generation

In [ ]:
num_users = 25
num_items = 80

interactions = []
for u in range(num_users):
    num_inter = np.random.randint(4, 12)
    items = np.random.choice(num_items, num_inter, replace=False)
    for i in items:
        interactions.append((u, i))

inter_df = pd.DataFrame(interactions, columns=['user_id', 'item_id'])

# Multi-mentor edges: (user_id, mentor_id, alpha)
multi_mentor_edges = [
    (0, 5, 0.65),
    (0, 7, 0.35),
    (1, 12, 0.80),
    (2, 5, 0.70),
    (3, 8, 0.90),
    (4, 15, 0.55),
    (4, 9, 0.45),
]

print(f"Created {len(inter_df)} user-item interactions")
print(f"Created {len(multi_mentor_edges)} mentor relationships (multi-mentor support)")

#### 2. Building the Heterogeneous Graph

In [ ]:
data = HeteroData()

data['user'].x = torch.randn(num_users, 16)
data['item'].x = torch.randn(num_items, 16)

# interact edges (user → item)
src = torch.tensor(inter_df['user_id'].values)
dst = torch.tensor(inter_df['item_id'].values)
data['user', 'interact', 'item'].edge_index = torch.stack([src, dst])

# mentor edges (user → mentor)
mentor_src = torch.tensor([u for u, m, a in multi_mentor_edges])
mentor_dst = torch.tensor([m for u, m, a in multi_mentor_edges])
mentor_alpha = torch.tensor([a for u, m, a in multi_mentor_edges], dtype=torch.float)

data['user', 'mentor', 'user'].edge_index = torch.stack([mentor_src, mentor_dst])
data['user', 'mentor', 'user'].edge_weight = mentor_alpha

print(data)

#### 3. Multi-Mentor GNN Model

In [ ]:
class MultiMentorGNN(torch.nn.Module):
    """
    Supports multiple mentors with different influence weights (alpha).
    """
    def __init__(self, hidden_dim=64, dropout=0.2):
        super().__init__()
        
        self.conv1 = HeteroConv({
            ('user', 'interact', 'item'): RGCNConv(16, hidden_dim, num_relations=1),
            ('user', 'mentor', 'user'): RGCNConv(16, hidden_dim, num_relations=1),
        })
        
        self.conv2 = HeteroConv({
            ('user', 'interact', 'item'): RGCNConv(hidden_dim, hidden_dim, num_relations=1),
            ('user', 'mentor', 'user'): RGCNConv(hidden_dim, hidden_dim, num_relations=1),
        })
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x_dict, edge_index_dict):
        x = self.conv1(x_dict, edge_index_dict)
        x = {k: F.relu(self.dropout(v)) for k, v in x.items()}
        
        x = self.conv2(x, edge_index_dict)
        x = {k: F.relu(self.dropout(v)) for k, v in x.items()}
        
        return x


model = MultiMentorGNN()
print("MultiMentorGNN model initialized")

#### 4. Recommendation Function with Multi-Mentor Support

In [ ]:
@torch.no_grad()
def get_recommendations(model, data, user_id, mentor_config=None, k=8):
    """
    mentor_config: list of tuples (mentor_id, alpha)
    Example: [(5, 0.7), (7, 0.3)]
    """
    model.eval()
    
    out = model({
        'user': data['user'].x,
        'item': data['item'].x
    }, {
        ('user', 'interact', 'item'): data['user', 'interact', 'item'].edge_index,
        ('user', 'mentor', 'user'): data['user', 'mentor', 'user'].edge_index
    })
    
    user_emb = out['user'][user_id].unsqueeze(0)
    item_embs = out['item']
    
    # Multi-Mentor Aggregation
    if mentor_config and len(mentor_config) > 0:
        mentor_ids = [m for m, a in mentor_config]
        alphas = [a for m, a in mentor_config]
        
        mentor_embs = out['user'][mentor_ids]
        
        final_emb = user_emb.clone()
        for m_emb, alpha in zip(mentor_embs, alphas):
            final_emb += alpha * (m_emb.unsqueeze(0) - user_emb)   # pull toward mentor
    else:
        final_emb = user_emb
    
    scores = torch.matmul(final_emb, item_embs.T).squeeze(0)
    topk_scores, topk_indices = torch.topk(scores, k=k)
    
    return {
        'user_id': user_id,
        'mentors': mentor_config,
        'recommended_items': topk_indices.tolist(),
        'scores': topk_scores.tolist()
    }


rec_base = get_recommendations(model, data, user_id=0, mentor_config=None)
print(f"User 0 (no mentors)     → {rec_base['recommended_items']}")

rec_single = get_recommendations(model, data, user_id=0, mentor_config=[(5, 0.8)])
print(f"User 0 + 1 mentor      → {rec_single['recommended_items']}")

rec_multi = get_recommendations(model, data, user_id=0, mentor_config=[(5, 0.65), (7, 0.35)])
print(f"User 0 + 2 mentors     → {rec_multi['recommended_items']}")